In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [43]:
import pandas as pd
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import (
    SentenceTransformer,
    util
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Milestone 2


In [44]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

letters = ["A","B","C","D","E"]

print(dataset)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [45]:
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(combine)

print("Answer 1 =", len(dataset[51]["combined_text"]))

Answer 1 = 614


2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [46]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Answer 2 =", tokenizer.vocab_size)

Answer 2 = 30522


3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [47]:
print("Answer 3 =", tokenizer.sep_token_id)

Answer 3 = 102


4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [48]:
prompts = [str(x) if x is not None else "" for x in dataset["prompt"]]

tokens = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Answer 4 =", tokens["input_ids"].shape)

Answer 4 = torch.Size([2000, 128])


5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [49]:
hidden = 768
heads = 12

print("Answer 5 =", hidden//heads)

Answer 5 = 64


6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [50]:
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(
    dataset[0]["prompt"],
    return_tensors="pt"
)

outputs = model(**inputs)

print("Answer 6 =", outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer 6 = torch.Size([1, 31, 768])


7.   Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [51]:
cls = outputs.last_hidden_state[0,0]

print("Answer 7 =", round(cls[:5].sum().item(),4))

Answer 7 = -1.2001


8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [52]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print(tokens)

fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1][0,0]

print("Answer 8 =", round(attention[0,fusion_index].item(),4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Answer 8 = 0.1025


9. Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [53]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt = dataset[0]["prompt"]

option = dataset[0]["B"]

emb = embedder.encode(
    [prompt,option],
    convert_to_tensor=True
)

score = util.cos_sim(
    emb[0],
    emb[1]
)

print("Answer 9 =", round(score.item(),4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer 9 = 0.7658


10. Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [54]:
tfidf_top3 = []

for _,row in df.iterrows():

    texts = [row["prompt"]]

    for l in letters:
        texts.append(row[l])

    vec = TfidfVectorizer()

    X = vec.fit_transform(texts)

    scores = cosine_similarity(
        X[0],
        X[1:]
    )[0]

    order = scores.argsort()[::-1]

    pred = [letters[i] for i in order[:3]]

    tfidf_top3.append(pred)

mini_top3 = []
for _,row in df.iterrows():

    texts = [row["prompt"]]

    for l in letters:
        texts.append(row[l])

    emb = embedder.encode(
        texts,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        emb[0],
        emb[1:]
    )[0]

    order = scores.argsort(descending=True)

    pred = [letters[int(i)] for i in order[:3]]

    mini_top3.append(pred)

score = 0

for pred,ans in zip(
    mini_top3,
    df.answer
):

    if ans==pred[0]:
        score+=1

    elif ans==pred[1]:
        score+=0.5

    elif ans==pred[2]:
        score+=1/3

map3 = score/len(df)

print("MAP@3 =",map3)

count=0

for tfidf,mini,ans in zip(
    tfidf_top3,
    mini_top3,
    df.answer
):

    if ans not in tfidf and ans in mini:
        count+=1

print("Answer 10")
print("MAP@3 =",map3)
print("Improved =",count)

MAP@3 = 0.42308333333333487
Answer 10
MAP@3 = 0.42308333333333487
Improved = 564


11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [55]:
classifier = pipeline(
    "zero-shot-classification"
)

row = df.iloc[1]

result = classifier(
    row["prompt"],
    candidate_labels=[
        row["A"],
        row["B"],
        row["C"]
    ]
)

print("Answer 11 =",round(result["scores"][0],4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Answer 11 = 0.4575


12. Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [56]:
softmax = classifier(
    row["prompt"],
    candidate_labels=[
        row["A"],
        row["B"],
        row["C"]
    ]
)

multi = classifier(
    row["prompt"],
    candidate_labels=[
        row["A"],
        row["B"],
        row["C"]
    ],
    multi_label=True
)

print(
    "Answer 12 =",
    abs(sum(softmax["scores"])-sum(multi["scores"]))
)

Answer 12 = 0.999490372636501


13. Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [57]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row = df.iloc[0]

text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Answer 13 =", answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Answer 13 = B
